In [25]:
import json
import ast
import time
import pandas as pd
from pathlib import Path
from openai import OpenAI
from tqdm import tqdm
import os

# ----------------------------
# File paths
# ----------------------------
CONFIG_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\config_prestep.ini")

ENV_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\.env")

INPUT_FILE = Path(
    r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058.xlsx"
)

OUTPUT_FILE = INPUT_FILE.with_name(INPUT_FILE.stem + "_llm_review_allrows.xlsx")

# UPDATE THIS IF NEEDED
KB_JSON_PATH = Path(
    r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\KnowledgeBase\All_HEALPAINCDEsDD_flattened.json"
)

MODEL_NAME = "gpt-4.1-mini"
CHECKPOINT_EVERY = 10

print("Paths loaded.")
print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_FILE:", OUTPUT_FILE)
print("KB_JSON_PATH:", KB_JSON_PATH)

Paths loaded.
INPUT_FILE: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058.xlsx
OUTPUT_FILE: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows.xlsx
KB_JSON_PATH: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\KnowledgeBase\All_HEALPAINCDEsDD_flattened.json


In [26]:
# ----------------------------
# Load API key from .env
# ----------------------------
def load_env_file(env_path):
    env_vars = {}
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_vars[key.strip()] = value.strip().strip('"').strip("'")
    return env_vars

env_vars = load_env_file(ENV_PATH)

api_key = env_vars.get("OPENAI_API_KEY") or env_vars.get("api_key")

if not api_key:
    raise ValueError("No OpenAI API key found in .env. Expected OPENAI_API_KEY or api_key.")

# ----------------------------
# Create OpenAI client
# ----------------------------
client = OpenAI(api_key=api_key)

print(f"Config file reference: {CONFIG_PATH}")
print(f".env loaded from: {ENV_PATH}")
print(f"Input file: {INPUT_FILE}")
print(f"Output file will be: {OUTPUT_FILE}")
print(f"API key loaded: {'Yes' if api_key else 'No'}")
print("OpenAI client created successfully.")

Config file reference: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\config_prestep.ini
.env loaded from: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\.env
Input file: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058.xlsx
Output file will be: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows.xlsx
API key loaded: Yes
OpenAI client created successfully.


In [27]:
# ----------------------------
# Load input workbook
# ----------------------------
df = pd.read_excel(INPUT_FILE).copy()

print("Input loaded.")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

# ----------------------------
# Column resolution helpers
# ----------------------------
def pick_first_existing(columns, candidates, required=False, label=None):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    if required:
        raise KeyError(f"Could not find required column for {label or candidates}. Tried: {candidates}")
    return None

resolved_cols = {
    "variable_name": pick_first_existing(
        df.columns,
        ["Variable / Field Name", "Variable Name", "name", "var"],
        required=True,
        label="variable_name"
    ),
    "form_name": pick_first_existing(
        df.columns,
        ["Form Name", "form_name", "form"],
        required=False,
        label="form_name"
    ),
    "question_text": pick_first_existing(
        df.columns,
        ["Field Label", "Question Text", "Description", "Variable Label"],
        required=False,
        label="question_text"
    ),
    "prestep_crf_match": pick_first_existing(
        df.columns,
        ["HEAL Core CRF Match"],
        required=False,
        label="prestep_crf_match"
    ),
    "prestep_confidence": pick_first_existing(
        df.columns,
        ["Prestep CRF Confidence", "Confidence Level"],
        required=False,
        label="prestep_confidence"
    ),
    "match_rationale": pick_first_existing(
        df.columns,
        ["Match Rationale", "Rationale"],
        required=False,
        label="match_rationale"
    ),
    "final_concept_match": pick_first_existing(
        df.columns,
        ["Best Match CDE Name", "Closest HEAL CDE Concept"],
        required=False,
        label="final_concept_match"
    ),
    "best_match_score": pick_first_existing(
        df.columns,
        ["Best Match Score", "Concept Match Score"],
        required=False,
        label="best_match_score"
    ),
    "best_match_crf": pick_first_existing(
        df.columns,
        ["Best Match CRF Name", "Concept Match CRF"],
        required=False,
        label="best_match_crf"
    ),
    "final_concept_status": pick_first_existing(
        df.columns,
        ["Concept Match Status", "Final Concept Status"],
        required=False,
        label="final_concept_status"
    ),
    "encoding_fidelity_score": pick_first_existing(
        df.columns,
        ["Encoding Fidelity Score"],
        required=False,
        label="encoding_fidelity_score"
    ),
    "potential_match_2": pick_first_existing(
        df.columns,
        ["Potential Match 2 - CDE Name", "Potential Match 2", "Potential Match 2 CDE Name"],
        required=False,
        label="potential_match_2"
    ),
    "potential_match_2_crf": pick_first_existing(
        df.columns,
        ["Potential Match 2 - CRF Name", "Potential Match 2 CRF Name"],
        required=False,
        label="potential_match_2_crf"
    ),
    "potential_match_3": pick_first_existing(
        df.columns,
        ["Potential Match 3 - CDE Name", "Potential Match 3", "Potential Match 3 CDE Name"],
        required=False,
        label="potential_match_3"
    ),
    "potential_match_3_crf": pick_first_existing(
        df.columns,
        ["Potential Match 3 - CRF Name", "Potential Match 3 CRF Name"],
        required=False,
        label="potential_match_3_crf"
    ),
    "best_match_source": pick_first_existing(
        df.columns,
        ["Best Match Source"],
        required=False,
        label="best_match_source"
    ),
    "protected_family_rule_applied": pick_first_existing(
        df.columns,
        ["Protected Family Rule Applied"],
        required=False,
        label="protected_family_rule_applied"
    ),
    "protected_family_expected_crf": pick_first_existing(
        df.columns,
        ["Protected Family Expected CRF"],
        required=False,
        label="protected_family_expected_crf"
    ),
    "blocked_primary_candidate": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate"],
        required=False,
        label="blocked_primary_candidate"
    ),
    "blocked_primary_candidate_score": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate Score"],
        required=False,
        label="blocked_primary_candidate_score"
    ),
    "blocked_primary_candidate_crf": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate CRF"],
        required=False,
        label="blocked_primary_candidate_crf"
    ),
    "blocked_primary_candidate_fidelity": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate Fidelity"],
        required=False,
        label="blocked_primary_candidate_fidelity"
    ),
    "protected_family_note": pick_first_existing(
        df.columns,
        ["Protected Family Note"],
        required=False,
        label="protected_family_note"
    ),
}

print("\nResolved columns:")
for k, v in resolved_cols.items():
    print(f"  {k}: {v}")

Input loaded.
Rows: 182
Columns: 46

Resolved columns:
  variable_name: Variable / Field Name
  form_name: Form Name
  question_text: Field Label
  prestep_crf_match: HEAL Core CRF Match
  prestep_confidence: Confidence Level
  match_rationale: Match Rationale
  final_concept_match: Best Match CDE Name
  best_match_score: Best Match Score
  best_match_crf: Best Match CRF Name
  final_concept_status: Concept Match Status
  encoding_fidelity_score: Encoding Fidelity Score
  potential_match_2: Potential Match 2 - CDE Name
  potential_match_2_crf: Potential Match 2 - CRF Name
  potential_match_3: Potential Match 3 - CDE Name
  potential_match_3_crf: Potential Match 3 - CRF Name
  best_match_source: Best Match Source
  protected_family_rule_applied: None
  protected_family_expected_crf: None
  blocked_primary_candidate: None
  blocked_primary_candidate_score: None
  blocked_primary_candidate_crf: None
  blocked_primary_candidate_fidelity: None
  protected_family_note: None


In [28]:
# ----------------------------
# Patch: resolve protected-family / blocked-proxy columns from v4 output
# Add this RIGHT AFTER Cell 3
# ----------------------------
resolved_cols.update({
    "best_match_source": pick_first_existing(
        df.columns,
        ["Best Match Source"],
        required=False,
        label="best_match_source"
    ),
    "protected_family_rule_applied": pick_first_existing(
        df.columns,
        ["Protected Family Rule Applied"],
        required=False,
        label="protected_family_rule_applied"
    ),
    "protected_family_expected_crf": pick_first_existing(
        df.columns,
        ["Protected Family Expected CRF"],
        required=False,
        label="protected_family_expected_crf"
    ),
    "blocked_primary_candidate": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate"],
        required=False,
        label="blocked_primary_candidate"
    ),
    "blocked_primary_candidate_score": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate Score"],
        required=False,
        label="blocked_primary_candidate_score"
    ),
    "blocked_primary_candidate_crf": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate CRF"],
        required=False,
        label="blocked_primary_candidate_crf"
    ),
    "blocked_primary_candidate_fidelity": pick_first_existing(
        df.columns,
        ["Blocked Primary Candidate Fidelity"],
        required=False,
        label="blocked_primary_candidate_fidelity"
    ),
    "protected_family_note": pick_first_existing(
        df.columns,
        ["Protected Family Note"],
        required=False,
        label="protected_family_note"
    ),
})

print("✅ Protected-family columns added to resolved_cols.")
for key in [
    "best_match_source",
    "protected_family_rule_applied",
    "protected_family_expected_crf",
    "blocked_primary_candidate",
    "blocked_primary_candidate_score",
    "blocked_primary_candidate_crf",
    "blocked_primary_candidate_fidelity",
    "protected_family_note",
]:
    print(f"  {key}: {resolved_cols.get(key)}")

✅ Protected-family columns added to resolved_cols.
  best_match_source: Best Match Source
  protected_family_rule_applied: None
  protected_family_expected_crf: None
  blocked_primary_candidate: None
  blocked_primary_candidate_score: None
  blocked_primary_candidate_crf: None
  blocked_primary_candidate_fidelity: None
  protected_family_note: None


In [29]:
# ----------------------------
# Candidate shortlist helpers
# PATCHED: carry CRF names from v4 output columns
# ----------------------------
def build_candidate_specs_from_row(row):
    """
    Pull shortlist candidates from the v4 output columns, including their CRF names.
    Preserves order and de-dupes by variable name.
    """
    candidate_pairs = [
        (resolved_cols["final_concept_match"], resolved_cols["best_match_crf"]),
        (resolved_cols["potential_match_2"], resolved_cols["potential_match_2_crf"]),
        (resolved_cols["potential_match_3"], resolved_cols["potential_match_3_crf"]),
    ]

    seen = set()
    specs = []

    for name_col, crf_col in candidate_pairs:
        if not name_col:
            continue

        candidate_name = norm_text(row.get(name_col, ""))
        candidate_crf = norm_text(row.get(crf_col, "")) if crf_col else ""

        if not candidate_name:
            continue

        key = candidate_name.lower()
        if key in seen:
            continue

        seen.add(key)
        specs.append({
            "candidate_name": candidate_name,
            "candidate_crf": candidate_crf
        })

    return specs


def build_official_candidate_packets(row):
    """
    Convert shortlisted candidate names into official KB packets.
    CRF names come from the v4 workbook columns, not from the flattened JSON.
    """
    candidate_specs = build_candidate_specs_from_row(row)
    packets = []

    for rank, spec in enumerate(candidate_specs, start=1):
        candidate_name = spec["candidate_name"]
        candidate_crf = spec["candidate_crf"]

        kb_entry = official_kb_by_varname.get(candidate_name.lower())
        if not kb_entry:
            continue

        packet = {
            "candidate_rank": rank,
            "candidate_from_row": candidate_name,
            "candidate_from_row_crf": candidate_crf,
            "kb_variable_name": kb_entry.get("kb_variable_name", candidate_name),
            "kb_crf_name": candidate_crf,
            **kb_entry
        }
        packets.append(packet)

    return packets


def build_official_candidate_names_json(row):
    packets = build_official_candidate_packets(row)
    names = [p["kb_variable_name"] for p in packets]
    return json.dumps(names, ensure_ascii=False)


def build_official_candidate_packets_json(row):
    packets = build_official_candidate_packets(row)
    return json.dumps(packets, ensure_ascii=False)


# Precompute shortlist columns
df["Official Candidate Names"] = df.apply(build_official_candidate_names_json, axis=1)
df["Official Candidate Packets JSON"] = df.apply(build_official_candidate_packets_json, axis=1)
df["Official Candidate Count"] = df["Official Candidate Packets JSON"].apply(
    lambda x: len(json.loads(x)) if norm_text(x) else 0
)

# Keep the v4 closest concept under the hardcoded review name
if resolved_cols["final_concept_match"]:
    df["vznhardcoded_closest_cde"] = df[resolved_cols["final_concept_match"]].fillna("")
else:
    df["vznhardcoded_closest_cde"] = ""

print("Official candidate shortlist columns created.")
display_cols = ["vznhardcoded_closest_cde", "Official Candidate Count"]
display(df[display_cols].head(10))

Official candidate shortlist columns created.


,vznhardcoded_closest_cde,Official Candidate Count
0,PEGAvgPainPastWeekScl,3
1,PEGPainPastWkEnjoyLifeScl,3
2,PEGPainPastWkGenrlActScl,3
3,PEGOverallScore,3
4,PROMISSleepWasRefreshScl,3
5,PROMISProblemWithSlpScl,3
6,PROMISDifficltFallAslpScl,3
7,PROMISSlpWasRestlessScl,3
8,PROMISTryHardGetToSlpScl,3
9,PROMISSlpDist6TotalScore,3


In [30]:
# ----------------------------
# Build row-level adjudication context packet
# PATCHED: explicit kb_crf_name in candidate packets
# ----------------------------
def build_row_level_adjudication_context_packet(row):
    packets_json = norm_text(row.get("Official Candidate Packets JSON", "[]"))
    try:
        official_candidate_packets = json.loads(packets_json) if packets_json else []
    except Exception:
        official_candidate_packets = []

    return {
        "row_context": {
            "variable_name": norm_text(row.get(resolved_cols["variable_name"], "")),
            "form_name": norm_text(row.get(resolved_cols["form_name"], "")) if resolved_cols["form_name"] else "",
            "question_text": norm_text(row.get(resolved_cols["question_text"], "")) if resolved_cols["question_text"] else "",
            "prestep_heal_core_crf_match": norm_text(row.get(resolved_cols["prestep_crf_match"], "")) if resolved_cols["prestep_crf_match"] else "",
            "prestep_crf_confidence": norm_text(row.get(resolved_cols["prestep_confidence"], "")) if resolved_cols["prestep_confidence"] else "",
            "prestep_match_rationale": norm_text(row.get(resolved_cols["match_rationale"], "")) if resolved_cols["match_rationale"] else "",
            "vznhardcoded_closest_cde": norm_text(row.get("vznhardcoded_closest_cde", "")),
            "best_match_score": norm_text(row.get(resolved_cols["best_match_score"], "")) if resolved_cols["best_match_score"] else "",
            "encoding_fidelity_score": norm_text(row.get(resolved_cols["encoding_fidelity_score"], "")) if resolved_cols["encoding_fidelity_score"] else "",
            "best_match_crf_name": norm_text(row.get(resolved_cols["best_match_crf"], "")) if resolved_cols["best_match_crf"] else "",
            "concept_match_status": norm_text(row.get(resolved_cols["final_concept_status"], "")) if resolved_cols["final_concept_status"] else "",
            "best_match_source": norm_text(row.get(resolved_cols["best_match_source"], "")) if resolved_cols.get("best_match_source") else "",
            "protected_family_rule_applied": norm_text(row.get(resolved_cols["protected_family_rule_applied"], "")) if resolved_cols.get("protected_family_rule_applied") else "",
            "protected_family_expected_crf": norm_text(row.get(resolved_cols["protected_family_expected_crf"], "")) if resolved_cols.get("protected_family_expected_crf") else "",
            "blocked_primary_candidate": norm_text(row.get(resolved_cols["blocked_primary_candidate"], "")) if resolved_cols.get("blocked_primary_candidate") else "",
            "blocked_primary_candidate_score": norm_text(row.get(resolved_cols["blocked_primary_candidate_score"], "")) if resolved_cols.get("blocked_primary_candidate_score") else "",
            "blocked_primary_candidate_crf": norm_text(row.get(resolved_cols["blocked_primary_candidate_crf"], "")) if resolved_cols.get("blocked_primary_candidate_crf") else "",
            "blocked_primary_candidate_fidelity": norm_text(row.get(resolved_cols["blocked_primary_candidate_fidelity"], "")) if resolved_cols.get("blocked_primary_candidate_fidelity") else "",
            "protected_family_note": norm_text(row.get(resolved_cols["protected_family_note"], "")) if resolved_cols.get("protected_family_note") else "",
            "official_candidate_count": int(row.get("Official Candidate Count", 0)),
        },
        "official_candidate_packets": official_candidate_packets
    }

print("✅ Row-level adjudication context packet builder patched with kb_crf_name.")

✅ Row-level adjudication context packet builder patched with kb_crf_name.


In [31]:
# ----------------------------
# Adjudication prompt + output parser
# STANDARDIZED decision_label
# PATCHED: instruct model to copy kb_crf_name exactly
# ----------------------------
ALLOWED_DECISION_LABELS = {
    "Accepted official KB match",
    "No confident official KB match",
    "Not adjudicated",
}

def normalize_decision_label(raw_label, best_best_match=""):
    label = norm_text(raw_label).lower()
    best_match = norm_text(best_best_match).lower()

    if not label:
        return "Not adjudicated"

    rejection_terms = {
        "reject",
        "rejected",
        "no confident official kb match",
        "no confident kb match",
        "no match",
        "not a confident match",
        "insufficient evidence",
    }
    if label in rejection_terms:
        return "No confident official KB match"

    acceptance_terms = {
        "accept",
        "accepted",
        "confident match",
        "strong match",
        "match",
        "official match",
    }
    if label in acceptance_terms:
        return "Accepted official KB match"

    if best_match and best_match != "no confident official kb match":
        return "Accepted official KB match"

    if best_match == "no confident official kb match":
        return "No confident official KB match"

    return "Not adjudicated"


ADJUDICATOR_SYSTEM_PROMPT = """
You are a highly conservative adjudicator for HEAL variable-level matching.

Your job:
- Review the row context packet.
- Review the official candidate packets.
- Choose the single BEST supported official KB match from the provided candidates only.
- If support is weak, ambiguous, or structurally inconsistent, return "No confident official KB match".

Important rules:
1. You may ONLY choose from the provided official candidate packets.
2. If the row has "No CRF match", treat that as a negative trust signal.
3. Low Best Match Score and low Encoding Fidelity Score should increase skepticism.
4. Prefer rejection over a weak or misleading official KB answer.
5. Each official candidate packet includes kb_variable_name and kb_crf_name.
6. If you choose a candidate, best_best_match must equal the chosen candidate's kb_variable_name.
7. If you choose a candidate, best_best_match_crf must equal the chosen candidate's kb_crf_name exactly.
8. Return JSON only. No markdown. No extra commentary.

Return exactly these keys:
- best_best_match
- best_best_match_crf
- decision_label
- primary_evidence_used
- adjudication_rationale
""".strip()


def parse_adjudication_output(raw_text):
    parsed = parse_json_object_from_text(raw_text)

    if not parsed:
        raise ValueError(f"Could not parse adjudication output as JSON.\nRaw output:\n{raw_text}")

    best_best_match = norm_text(parsed.get("best_best_match", "No confident official KB match"))
    raw_decision_label = norm_text(parsed.get("decision_label", ""))

    standardized_decision_label = normalize_decision_label(
        raw_label=raw_decision_label,
        best_best_match=best_best_match
    )

    return {
        "best_best_match": best_best_match,
        "best_best_match_crf": norm_text(parsed.get("best_best_match_crf", "")),
        "decision_label": standardized_decision_label,
        "primary_evidence_used": norm_text(parsed.get("primary_evidence_used", "")),
        "adjudication_rationale": norm_text(parsed.get("adjudication_rationale", "")),
        "raw_adjudication_json": raw_text,
    }

print("Adjudication prompt + parser loaded.")
print("Allowed decision labels:", sorted(ALLOWED_DECISION_LABELS))

Adjudication prompt + parser loaded.
Allowed decision labels: ['Accepted official KB match', 'No confident official KB match', 'Not adjudicated']


In [32]:
# ----------------------------
# Adjudicate one row with LLM
# PATCHED: deterministically backfill best_best_match_crf from candidate packets
# ----------------------------
def resolve_candidate_crf_from_payload(best_best_match, payload):
    """
    Deterministically recover the CRF name for the chosen candidate
    from the official candidate packets.
    """
    chosen = norm_text(best_best_match)
    if not chosen or chosen == "No confident official KB match":
        return ""

    packets = payload.get("official_candidate_packets", [])
    for packet in packets:
        packet_var = norm_text(packet.get("kb_variable_name", ""))
        if packet_var.lower() == chosen.lower():
            return norm_text(packet.get("kb_crf_name", ""))

    return ""


def adjudicate_row_with_llm(row):
    payload = build_row_level_adjudication_context_packet(row)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": ADJUDICATOR_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False, indent=2)},
        ],
        temperature=0
    )

    raw_content = response.choices[0].message.content

    if not raw_content:
        raise ValueError("Model returned empty content.")

    parsed = parse_adjudication_output(raw_content)

    # Deterministic CRF backfill / override from payload
    resolved_crf = resolve_candidate_crf_from_payload(
        best_best_match=parsed["best_best_match"],
        payload=payload
    )

    if parsed["best_best_match"] == "No confident official KB match":
        parsed["best_best_match_crf"] = ""
    elif resolved_crf:
        parsed["best_best_match_crf"] = resolved_crf

    return {
        **parsed,
        "review_status": "Reviewed by LLM",
        "adjudication_attempts": 1,
        "adjudication_error": ""
    }


def adjudicate_row_with_retry(row, max_retries=3, base_sleep_seconds=2):
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            result = adjudicate_row_with_llm(row)
            result["adjudication_attempts"] = attempt
            return result

        except Exception as e:
            last_error = str(e)
            print(f"[warning] adjudication failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "best_best_match": "No confident official KB match",
        "best_best_match_crf": "",
        "decision_label": "No confident official KB match",
        "primary_evidence_used": "llm_error_fallback",
        "adjudication_rationale": "",
        "raw_adjudication_json": "",
        "review_status": "LLM error fallback",
        "adjudication_attempts": max_retries,
        "adjudication_error": last_error
    }

print("LLM adjudication + retry wrapper loaded.")
print("best_best_match_crf will now be backfilled from official candidate packets.")

LLM adjudication + retry wrapper loaded.
best_best_match_crf will now be backfilled from official candidate packets.


In [33]:
# ----------------------------
# Adjudicate one row with LLM
# ----------------------------
def adjudicate_row_with_llm(row):
    payload = build_row_level_adjudication_context_packet(row)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": ADJUDICATOR_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False, indent=2)},
        ],
        temperature=0
    )

    raw_content = response.choices[0].message.content

    if not raw_content:
        raise ValueError("Model returned empty content.")

    parsed = parse_adjudication_output(raw_content)

    return {
        **parsed,
        "review_status": "Reviewed",
        "adjudication_attempts": 1,
        "adjudication_error": ""
    }

def adjudicate_row_with_retry(row, max_retries=3, base_sleep_seconds=2):
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            result = adjudicate_row_with_llm(row)
            result["adjudication_attempts"] = attempt
            return result

        except Exception as e:
            last_error = str(e)
            print(f"[warning] adjudication failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "best_best_match": "No confident official KB match",
        "best_best_match_crf": "",
        "decision_label": "No confident official KB match",
        "primary_evidence_used": "llm_error_fallback",
        "adjudication_rationale": "",
        "raw_adjudication_json": "",
        "review_status": "ERROR",
        "adjudication_attempts": max_retries,
        "adjudication_error": last_error
    }

print("LLM adjudication + retry wrapper loaded.")

LLM adjudication + retry wrapper loaded.


In [34]:
# ----------------------------
# Weak-evidence veto + post-override safeguards
# STANDARDIZED decision_label
# ----------------------------
PRE_ADJUDICATION_RULES = {
    "no_crf_value": "No CRF match",
    "max_best_match_score_for_veto": 70.0,
    "max_encoding_fidelity_for_veto": 60.0
}

def has_no_crf_anchor(row):
    col = resolved_cols["prestep_crf_match"]
    value = norm_text(row.get(col, "")) if col else ""
    return value == PRE_ADJUDICATION_RULES["no_crf_value"] or value == ""

def is_weak_unanchored_row(row):
    best_match_col = resolved_cols["best_match_score"]
    encoding_col = resolved_cols["encoding_fidelity_score"]

    best_match_score = safe_float(row.get(best_match_col), default=None) if best_match_col else None
    encoding_fidelity = safe_float(row.get(encoding_col), default=None) if encoding_col else None

    if best_match_score is None or encoding_fidelity is None:
        return False

    return (
        has_no_crf_anchor(row)
        and best_match_score <= PRE_ADJUDICATION_RULES["max_best_match_score_for_veto"]
        and encoding_fidelity <= PRE_ADJUDICATION_RULES["max_encoding_fidelity_for_veto"]
    )

def build_no_confident_kb_result(
    reason,
    review_status="Skipped - weak evidence veto",
    primary_evidence="deterministic_veto_gate"
):
    return {
        "best_best_match": "No confident official KB match",
        "best_best_match_crf": "",
        "decision_label": "No confident official KB match",
        "primary_evidence_used": primary_evidence,
        "adjudication_rationale": reason,
        "raw_adjudication_json": "",
        "review_status": review_status,
        "adjudication_attempts": 0,
        "adjudication_error": ""
    }

def build_not_adjudicated_result(
    reason="Row was not adjudicated.",
    review_status="Not adjudicated",
    primary_evidence="not_adjudicated"
):
    return {
        "best_best_match": "",
        "best_best_match_crf": "",
        "decision_label": "Not adjudicated",
        "primary_evidence_used": primary_evidence,
        "adjudication_rationale": reason,
        "raw_adjudication_json": "",
        "review_status": review_status,
        "adjudication_attempts": 0,
        "adjudication_error": ""
    }

def apply_post_adjudication_override_if_needed(row, result):
    """
    Secondary safety net in case a weak unanchored row somehow survives.
    """
    if is_weak_unanchored_row(row) and result.get("decision_label", "") != "No confident official KB match":
        reason = (
            "LLM result was overridden because the row had no usable HEAL Core CRF anchor, "
            f"Best Match Score={norm_text(row.get(resolved_cols['best_match_score'], '')) if resolved_cols['best_match_score'] else ''}, "
            f"Encoding Fidelity Score={norm_text(row.get(resolved_cols['encoding_fidelity_score'], '')) if resolved_cols['encoding_fidelity_score'] else ''}, "
            "and failed the weak-evidence safeguard."
        )
        return build_no_confident_kb_result(
            reason=reason,
            review_status="Overridden - weak evidence veto",
            primary_evidence="post_adjudication_override"
        )

    # final normalization sweep, just in case
    result = result.copy()
    result["decision_label"] = normalize_decision_label(
        raw_label=result.get("decision_label", ""),
        best_best_match=result.get("best_best_match", "")
    )
    return result

print("Weak-evidence veto + post-override helpers loaded.")
print("Deterministic outputs now use only the standardized decision labels.")

Weak-evidence veto + post-override helpers loaded.
Deterministic outputs now use only the standardized decision labels.


In [35]:
# ----------------------------
# Protected-family blocked-proxy helpers
# Add this RIGHT AFTER Cell 9
# ----------------------------
def yes_like(value):
    return norm_text(value).lower() in {"yes", "y", "true", "1"}

def is_protected_family_blocked_row(row):
    """
    Detect rows where v4 intentionally blocked a cross-family proxy from
    becoming the primary best match.
    """
    protected_flag_col = resolved_cols.get("protected_family_rule_applied")
    blocked_candidate_col = resolved_cols.get("blocked_primary_candidate")
    best_match_source_col = resolved_cols.get("best_match_source")

    protected_flag = norm_text(row.get(protected_flag_col, "")) if protected_flag_col else ""
    blocked_candidate = norm_text(row.get(blocked_candidate_col, "")) if blocked_candidate_col else ""
    best_match_source = norm_text(row.get(best_match_source_col, "")) if best_match_source_col else ""

    return (
        yes_like(protected_flag)
        and (
            blocked_candidate != ""
            or best_match_source == "Protected-family proxy blocked"
        )
    )

def build_protected_family_not_adjudicated_result(row):
    """
    Deterministic result for rows that should NOT be adjudicated because
    the upstream v4 workflow already blocked a cross-family proxy candidate.
    """
    blocked_name = norm_text(row.get(resolved_cols["blocked_primary_candidate"], "")) if resolved_cols.get("blocked_primary_candidate") else ""
    blocked_crf = norm_text(row.get(resolved_cols["blocked_primary_candidate_crf"], "")) if resolved_cols.get("blocked_primary_candidate_crf") else ""
    blocked_score = norm_text(row.get(resolved_cols["blocked_primary_candidate_score"], "")) if resolved_cols.get("blocked_primary_candidate_score") else ""
    blocked_fidelity = norm_text(row.get(resolved_cols["blocked_primary_candidate_fidelity"], "")) if resolved_cols.get("blocked_primary_candidate_fidelity") else ""
    protected_note = norm_text(row.get(resolved_cols["protected_family_note"], "")) if resolved_cols.get("protected_family_note") else ""
    expected_crf = norm_text(row.get(resolved_cols["protected_family_expected_crf"], "")) if resolved_cols.get("protected_family_expected_crf") else ""

    reason = (
        f"Row was not adjudicated because upstream v4 flagged it as a protected-family proxy case. "
        f"Expected protected CRF='{expected_crf}'. "
        f"Blocked proxy candidate='{blocked_name}' from CRF='{blocked_crf}' "
        f"(Concept Score={blocked_score}, Fidelity={blocked_fidelity}). "
        f"{protected_note}"
    ).strip()

    return build_not_adjudicated_result(
        reason=reason,
        review_status="Protected-family manual review",
        primary_evidence="protected_family_proxy_blocked"
    )

print("✅ Protected-family blocked-proxy helpers loaded.")

✅ Protected-family blocked-proxy helpers loaded.


In [36]:
# ----------------------------
# Run adjudication on eligible rows with checkpoint saving
# FINAL standardized decision_label sweep
# PATCHED for protected-family blocked proxies
# ----------------------------
status_col = resolved_cols["final_concept_status"]

# Identify protected-family rows that should never be sent to the LLM
protected_blocked_mask = df.apply(is_protected_family_blocked_row, axis=1)
protected_blocked_indices = df[protected_blocked_mask].index.tolist()

print(f"Protected-family rows blocked from adjudication: {len(protected_blocked_indices)}")

# Base eligibility
if status_col:
    base_eligible_mask = (
        df[status_col].isin({"High concept match", "Possible concept match"}) &
        (df["Official Candidate Count"] > 0)
    )
else:
    base_eligible_mask = df["Official Candidate Count"] > 0

# Remove protected blocked rows from LLM eligibility
eligible_mask = base_eligible_mask & (~protected_blocked_mask)
eligible_indices = df[eligible_mask].index.tolist()

print(f"Eligible rows for adjudication: {len(eligible_indices)}")

# Initialize output columns
output_cols = [
    "vznhardcoded_closest_cde",
    "best_best_match",
    "best_best_match_crf",
    "decision_label",
    "primary_evidence_used",
    "adjudication_rationale",
    "raw_adjudication_json",
    "review_status",
    "adjudication_attempts",
    "adjudication_error"
]

for col in output_cols:
    if col not in df.columns:
        df[col] = ""

checkpoint_file = OUTPUT_FILE.with_name(OUTPUT_FILE.stem + "_checkpoint.xlsx")

processed_count = 0
vetoed_count = 0
llm_reviewed_count = 0
overridden_count = 0
protected_manual_review_count = 0

# ------------------------------------------------
# First: deterministically populate protected-family blocked rows
# ------------------------------------------------
for idx in protected_blocked_indices:
    row = df.loc[idx]
    result = build_protected_family_not_adjudicated_result(row)

    df.at[idx, "best_best_match"] = result["best_best_match"]
    df.at[idx, "best_best_match_crf"] = result["best_best_match_crf"]
    df.at[idx, "decision_label"] = result["decision_label"]
    df.at[idx, "primary_evidence_used"] = result["primary_evidence_used"]
    df.at[idx, "adjudication_rationale"] = result["adjudication_rationale"]
    df.at[idx, "raw_adjudication_json"] = result["raw_adjudication_json"]
    df.at[idx, "review_status"] = result["review_status"]
    df.at[idx, "adjudication_attempts"] = result["adjudication_attempts"]
    df.at[idx, "adjudication_error"] = result["adjudication_error"]

    protected_manual_review_count += 1

# ------------------------------------------------
# Then: normal adjudication loop
# ------------------------------------------------
for row_num, idx in enumerate(tqdm(eligible_indices, desc="Adjudicating eligible rows"), start=1):
    row = df.loc[idx]

    # ----------------------------
    # Deterministic pre-adjudication weak-evidence veto
    # ----------------------------
    if is_weak_unanchored_row(row):
        reason = (
            "Row was not sent to LLM adjudication because it had no usable HEAL Core CRF anchor, "
            f"Best Match Score={norm_text(row.get(resolved_cols['best_match_score'], '')) if resolved_cols['best_match_score'] else ''}, "
            f"Encoding Fidelity Score={norm_text(row.get(resolved_cols['encoding_fidelity_score'], '')) if resolved_cols['encoding_fidelity_score'] else ''}, "
            "and failed the weak-evidence pre-adjudication gate."
        )
        result = build_no_confident_kb_result(reason)
        vetoed_count += 1

    else:
        result = adjudicate_row_with_retry(row, max_retries=3, base_sleep_seconds=2)
        llm_reviewed_count += 1

        # ----------------------------
        # Post-adjudication override safeguard
        # ----------------------------
        maybe_overridden = apply_post_adjudication_override_if_needed(row, result)
        if maybe_overridden.get("review_status") == "Overridden - weak evidence veto":
            overridden_count += 1
        result = maybe_overridden

    # final per-row normalization
    result["decision_label"] = normalize_decision_label(
        raw_label=result.get("decision_label", ""),
        best_best_match=result.get("best_best_match", "")
    )

    # write results back
    df.at[idx, "best_best_match"] = result["best_best_match"]
    df.at[idx, "best_best_match_crf"] = result["best_best_match_crf"]
    df.at[idx, "decision_label"] = result["decision_label"]
    df.at[idx, "primary_evidence_used"] = result["primary_evidence_used"]
    df.at[idx, "adjudication_rationale"] = result["adjudication_rationale"]
    df.at[idx, "raw_adjudication_json"] = result["raw_adjudication_json"]
    df.at[idx, "review_status"] = result["review_status"]
    df.at[idx, "adjudication_attempts"] = result["adjudication_attempts"]
    df.at[idx, "adjudication_error"] = result["adjudication_error"]

    processed_count += 1

    if processed_count % CHECKPOINT_EVERY == 0:
        df.to_excel(checkpoint_file, index=False)
        print(f"\nCheckpoint saved after {processed_count} rows: {checkpoint_file}")

# ----------------------------
# Final workbook-wide normalization sweep
# ----------------------------

# Normalize all populated decision labels
df["decision_label"] = df.apply(
    lambda row: normalize_decision_label(
        raw_label=row.get("decision_label", ""),
        best_best_match=row.get("best_best_match", "")
    ),
    axis=1
)

# Any row that never got adjudication output becomes "Not adjudicated"
blank_decision_mask = df["decision_label"].astype(str).str.strip().eq("")
df.loc[blank_decision_mask, "decision_label"] = "Not adjudicated"

blank_review_mask = df["review_status"].astype(str).str.strip().eq("")
df.loc[blank_review_mask, "review_status"] = "Not adjudicated"

# Save final output
df.to_excel(OUTPUT_FILE, index=False)

print("\nAdjudication complete.")
print(f"Final reviewed file saved to: {OUTPUT_FILE}")
print(f"Checkpoint file path: {checkpoint_file}")
print(f"Protected-family manual review rows: {protected_manual_review_count}")
print(f"Rows vetoed before LLM adjudication: {vetoed_count}")
print(f"Rows reviewed by LLM: {llm_reviewed_count}")
print(f"Rows overridden after LLM adjudication: {overridden_count}")

print("\nFinal decision_label counts:")
print(df["decision_label"].value_counts(dropna=False))

Protected-family rows blocked from adjudication: 0
Eligible rows for adjudication: 89


Adjudicating eligible rows:  11%|█         | 10/89 [00:41<05:40,  4.31s/it]


Checkpoint saved after 10 rows: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows_checkpoint.xlsx


Adjudicating eligible rows:  22%|██▏       | 20/89 [01:21<04:34,  3.97s/it]


Checkpoint saved after 20 rows: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows_checkpoint.xlsx


...


Checkpoint saved after 70 rows: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows_checkpoint.xlsx


Adjudicating eligible rows:  90%|████████▉ | 80/89 [05:13<00:19,  2.13s/it]


Checkpoint saved after 80 rows: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows_checkpoint.xlsx


Adjudicating eligible rows: 100%|██████████| 89/89 [05:21<00:00,  3.61s/it]



Adjudication complete.
Final reviewed file saved to: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows.xlsx
Checkpoint file path: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058_llm_review_allrows_checkpoint.xlsx
Protected-family manual review rows: 0
Rows vetoed before LLM adjudication: 14
Rows reviewed by LLM: 75
Rows overridden after LLM adjudication: 0

Final decision_label counts:
decision_label
Not adjudicated                   93
Accepted official KB match        65
No confident official KB match    24
Name: count, dtype: int64


In [ ]:
# ----------------------------
# Quick QC checks
# ----------------------------
print("\nReview status counts:")
print(df["review_status"].value_counts(dropna=False))

print("\nDecision label counts:")
print(df["decision_label"].value_counts(dropna=False).head(20))

qc_cols = [
    resolved_cols["variable_name"],
    resolved_cols["prestep_crf_match"],
    resolved_cols["best_match_score"],
    resolved_cols["encoding_fidelity_score"],
    "vznhardcoded_closest_cde",
    "best_best_match",
    "review_status"
]

qc_cols = [c for c in qc_cols if c is not None]

display(df[qc_cols].head(20))